In [1]:
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import regionprops_table
from skimage.filters import threshold_otsu
from skimage.exposure import rescale_intensity
from skimage.util import img_as_ubyte, img_as_float
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from tqdm import tqdm
from skimage import io
import os
from skimage.morphology import binary_dilation, binary_erosion, disk

In [2]:
mask = tifffile.imread(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\composite_segmentation\Composite_cp_masks.tif")

In [3]:
cell_props = regionprops_table(mask, properties=["label", "area", "centroid"])

In [4]:
cell_props_df = pd.DataFrame(cell_props)
thre = threshold_otsu(cell_props_df["area"])
cell_props_df = cell_props_df[cell_props_df["area"] > thre]

c:\Users\zfang38\AppData\Local\anaconda3\envs\skim2\lib\site-packages\skimage\filters\thresholding.py:329: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  first_pixel = image.ravel()[0]
c:\Users\zfang38\AppData\Local\anaconda3\envs\skim2\lib\site-packages\skimage\filters\thresholding.py:277: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  image.ravel(), nbins, source_range='image'


In [5]:
cell_props_df['label'].unique().shape

(1817,)

In [6]:
valid_labels = cell_props_df['label'].unique()
valid_mask = np.isin(mask, valid_labels)
mask[~valid_mask] = 0

In [7]:
np.unique(mask).shape

(1818,)

In [8]:
# # relabel the mask
# from skimage.measure import label
# mask = label(mask)

In [9]:
tifffile.imwrite(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\composite_segmentation\Composite_cp_masks_cleaned.tif", mask.astype(np.uint16))

In [10]:
io.imsave(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\composite_segmentation\Composite_cp_masks_cleaned.png", mask.astype(np.uint16))

In [11]:
dapi = tifffile.imread(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\nuclei_segmentation\C1_01_msg_83_ssa+_empty_ro60_ro52_cp_masks.tif")

In [12]:
dapi_binary = dapi > 0
nuclei = binary_erosion(dapi_binary, disk(3))
dilated_mask = binary_dilation(dapi_binary, disk(7))
cytosol = dilated_mask.astype(int) - dapi_binary.astype(int)

In [13]:
tifffile.imwrite(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\nuclei_segmentation\Composite_cytosol.tif", cytosol.astype(np.uint16))
tifffile.imwrite(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\nuclei_segmentation\dapi_binary.tif", nuclei.astype(np.uint16))

In [14]:
ro52 = tifffile.imread(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_registered\registered\C4\C4_01_msg_83_ssa+_empty_ro60_ro52.tif")
ro52 = img_as_float(ro52)
ro52 = rescale_intensity(ro52, out_range=(0, 1))
nuclei_ro52 = ro52 * nuclei
cytosol_ro52 = ro52 * cytosol
ro60 = tifffile.imread(r"Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_registered\registered\C3\C3_01_msg_83_ssa+_empty_ro60_ro52.tif")
ro60 = img_as_float(ro60)
ro60 = rescale_intensity(ro60, out_range=(0, 1))
nuclei_ro60 = ro60 * nuclei
cytosol_ro60 = ro60 * cytosol

In [15]:
def masked_mean(mask, intensity_image):
    # Average of all the non-zero pixels in the mask
    temp = intensity_image[mask > 0]
    mean = np.sum(temp) / np.sum(temp > 0)
    return mean

In [16]:
nuclei_ro52_exp = regionprops_table(mask, properties=["label"], extra_properties=[masked_mean], intensity_image=nuclei_ro52)
nuclei_ro60_exp = regionprops_table(mask, properties=["label"], extra_properties=[masked_mean], intensity_image=nuclei_ro60)
cytosol_ro52_exp = regionprops_table(mask, properties=["label"], extra_properties=[masked_mean], intensity_image=cytosol_ro52)
cytosol_ro60_exp = regionprops_table(mask, properties=["label"], extra_properties=[masked_mean], intensity_image=cytosol_ro60)

In [17]:
# plot figures

In [18]:
# cytosol_ro52_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
# cytosol_ro60_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
# nuclei_ro52_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
# nuclei_ro60_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

In [19]:
# v1 = np.max(cytosol_ro52_exp['masked_mean'])
# v2 = np.max(cytosol_ro60_exp['masked_mean'])
# v3 = np.max(nuclei_ro52_exp['masked_mean'])
# v4 = np.max(nuclei_ro60_exp['masked_mean'])
# norm_ro52 = Normalize(vmin=0, vmax=max(v1, v3))
# norm_ro60 = Normalize(vmin=0, vmax=max(v2, v4))
# for label in tqdm(np.unique(mask)):
#     if label == 0:
#         continue
#     mask_label = mask == label


#     cmap = cm.get_cmap('viridis')
#     cytosol_ro52_rgb[mask_label] = cmap(norm_ro52(cytosol_ro52_exp['masked_mean'][np.where(cytosol_ro52_exp['label'] == label)])).flatten()[:3] * 255
#     cytosol_ro60_rgb[mask_label] = cmap(norm_ro60(cytosol_ro60_exp['masked_mean'][np.where(cytosol_ro60_exp['label'] == label)])).flatten()[:3] * 255
#     nuclei_ro52_rgb[mask_label] = cmap(norm_ro52(nuclei_ro52_exp['masked_mean'][np.where(nuclei_ro52_exp['label'] == label)])).flatten()[:3] * 255
#     nuclei_ro60_rgb[mask_label] = cmap(norm_ro60(nuclei_ro60_exp['masked_mean'][np.where(nuclei_ro60_exp['label'] == label)])).flatten()[:3] * 255

In [20]:
# out_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20250123_ssa_scan\ssa-_126-2\00_region_segmentation\figures'
# os.makedirs(out_dir, exist_ok=True)

In [21]:
# io.imsave(os.path.join(out_dir, 'cytosl_ro52_rgb.png'), cytosol_ro52_rgb.astype(np.uint8))
# io.imsave(os.path.join(out_dir, 'cytosol_ro60_rgb.png'), cytosol_ro60_rgb.astype(np.uint8))
# io.imsave(os.path.join(out_dir, 'nuclei_ro52_rgb.png'), nuclei_ro52_rgb.astype(np.uint8))
# io.imsave(os.path.join(out_dir, 'nuclei_ro60_rgb.png'), nuclei_ro60_rgb.astype(np.uint8))

In [22]:
results_dir = r'Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_region_segmentation\results'
os.makedirs(results_dir, exist_ok=True)

In [23]:
results = pd.DataFrame({
    'label': cell_props_df['label'],
    'area': cell_props_df['area'],
    'row': cell_props_df['centroid-0'],
    'col': cell_props_df['centroid-1'],
    'cytosol_ro52_mean': cytosol_ro52_exp['masked_mean'],
    'cytosol_ro60_mean': cytosol_ro60_exp['masked_mean'],
    'nuclei_ro52_mean': nuclei_ro52_exp['masked_mean'],
    'nuclei_ro60_mean': nuclei_ro60_exp['masked_mean'],
})

In [24]:
results.to_csv(os.path.join(results_dir, 'compartment_properties.csv'), index=False)

In [25]:
# Set all images to None
dapi = None
ro52 = None
nuclei_ro52 = None
cytosol_ro52 = None
ro60 = None
nuclei_ro60 = None
cytosol_ro60 = None
dilated_mask = None

# other markers

In [26]:
from functools import reduce

In [27]:
registered_dir = r'Y:\coskun-lab\Zhou\12_MSG\msg_germinal_centers\IF\20240808_multiplexed_IF\SSA+_83\00_registered\registered'
channels_l = os.listdir(registered_dir)
channels_l.sort()
channels_l = channels_l[1:]

In [28]:
channels_l

['C2', 'C3', 'C4']

In [29]:
im_l = os.listdir(os.path.join(registered_dir, channels_l[0]))
im_l.sort()
im_l = [im for im in im_l if im.endswith('.tif')]
fn = im_l[0].split('.')[0]
x1 = 5
x2 = 8
markers = fn.split('_')[x1:x2]
markers

['empty', 'ro60', 'ro52']

In [30]:
# Whole image
intensities = []
for c in channels_l:
    im_l = os.listdir(os.path.join(registered_dir, c))
    im_l.sort()
    for im in tqdm(im_l):
        if im.endswith('.tif'):
            fn = im.split('.')[0]
            markers = fn.split('_')[x1:x2]
            marker = markers[int(c[1])-2]
            if marker == 'empty' or marker == 'ro52' or marker == 'ro60':
                continue
            intensity = tifffile.imread(os.path.join(registered_dir, c, im))
            intensity = img_as_float(intensity)
            intensity = rescale_intensity(intensity, out_range=(0, 1))
            exp = regionprops_table(mask, intensity, properties=['label', 'mean_intensity'])
            exp = pd.DataFrame(exp)
            exp.rename(columns={'mean_intensity': marker.upper()}, inplace=True)
            intensities.append(exp)

100%|██████████| 9/9 [03:23<00:00, 22.67s/it]


In [31]:
merged_df = reduce(lambda left, right: pd.merge(left, right, on='label'), intensities)

In [32]:
merged_df.to_csv(os.path.join(results_dir, 'full_markerset_expression.csv'), index=False)

# relabel mask

In [1]:
from skimage.segmentation import relabel_sequential

In [4]:
mask = tifffile.imread(r"Y:\coskun-lab2\Zhou\12_MSG\20250123_ssa_scan\ssa-_126-1\00_region_segmentation\composite_segmentation\Composite_cp_masks_cleaned.tif")

In [5]:
new_labels, _, _ = relabel_sequential(mask)
tifffile.imwrite(r"Y:\coskun-lab2\Zhou\12_MSG\20250123_ssa_scan\ssa-_126-1\00_region_segmentation\composite_segmentation\Composite_cp_masks_cleaned_relabelled.tif", new_labels.astype(np.uint16))

In [11]:
from skimage.color import label2rgb
mask_rgb = label2rgb(new_labels, bg_label=0, colors=cm.viridis(np.linspace(0, 1, 15)), alpha=0.5)

In [12]:
tifffile.imwrite(r"Y:\coskun-lab2\Zhou\12_MSG\20250123_ssa_scan\ssa-_126-1\00_region_segmentation\composite_segmentation\Composite_cp_masks_cleaned_relabelled_rgb.tif", (mask_rgb * 255).astype(np.uint8))